# Modeling

Huấn luyện và đánh giá mô hình với 4 thiết kế thí nghiệm (α₁→α₄).

| Thành phần | Chi tiết |
|------------|----------|
| **4 models** | LGBM, XGB, CatBoost, KNN (baseline) |
| **Stratified K-Fold** | 5 folds, đảm bảo tỷ lệ lớp ổn định |
| **SMOTE + RandomUnderSampler** | Oversample minority → 25%, undersample majority → 50% |
| **GridSearchCV** | Tối ưu hyperparameter với F1 scoring |
| **Metrics** | Precision, Recall, F1, ROC-AUC, Confusion Matrix |
| **4 experiments** | α₁ retroactive, α₂ realistic, α₃ combined, α₄ incremental |


## 3.1 Setup

Import các thư viện cần thiết và load config.
Các mô hình chính: LightGBM, XGBoost, CatBoost, và KNN làm baseline đơn giản.


In [ ]:
"""Configuration constants for DS108 Lab 4 pipeline."""
import os

# Paths
DATA_DIR = "Data"
RESULTS_DIR = "results"
EDA_DIR = os.path.join(RESULTS_DIR, "eda")
EXP_DIR = os.path.join(RESULTS_DIR, "experiments")
MODEL_DIR = os.path.join(RESULTS_DIR, "models")
PLOT_DIR = os.path.join(RESULTS_DIR, "plots")

# Raw files
DELAY_46 = os.path.join(DATA_DIR, "delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv")
NOT_DELAY_46 = os.path.join(DATA_DIR, "not_delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv")
DELAY_79 = os.path.join(DATA_DIR, "delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv")
NOT_DELAY_79 = os.path.join(DATA_DIR, "not_delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv")

# Experiment labels
PERIOD_A = "7_9"   # future
PERIOD_B = "4_6"   # past

# Random seed for reproducibility
RANDOM_STATE = 42

# Train/Val/Test split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Stratified K-Fold
N_SPLITS = 5

# Incremental learning ratios for alpha_4
INCREMENTAL_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]

# Model light hyperparameters (tuned via CV inside training)
LGBM_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_estimators": 1000,
    "random_state": RANDOM_STATE,
    "is_unbalance": True,
}

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "n_estimators": 1000,
    "random_state": RANDOM_STATE,
}

CAT_PARAMS = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "auto_class_weights": "Balanced",
}

KNN_PARAMS = {
    "n_neighbors": 5,
}

        import pandas as pd
        import numpy as np
        import os
        import config
        from sklearn.preprocessing import StandardScaler, LabelEncoder
        from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
        from sklearn.metrics import (
            accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
            classification_report, confusion_matrix
        )
        from sklearn.neighbors import KNeighborsClassifier
        import lightgbm as lgb
        import xgboost as xgb
        from catboost import CatBoostClassifier

## 3.2 Load & Preprocess Data

Hàm tiện ích để load dữ liệu đã qua preprocessing và chuẩn bị cho modeling.

> Lưu ý quan trọng: α₃ (K-Fold Combined) dùng toàn bộ dữ liệu đã merge, không cần split train/test riêng. Tránh preprocess test set 2 lần.


In [ ]:
def load_and_preprocess():
    df_46 = pd.read_csv("processed/tidy_4_6.csv", low_memory=False)
    df_79 = pd.read_csv("processed/tidy_7_9.csv", low_memory=False)

    # Basic cleaning
    for df in [df_46, df_79]:
        # Drop leakage/ID columns if still present
        drop_cols = ["Order date", "VSD", "GLOBAL_NO", "CUST_CD",
                     "PRODUCT_CD", "SUPPLIER_CD", "SHIP_DECISION_NO",
                     "SOUF_RCV_NO", "QTUF_RCV_NO", "REASON_CD"]
        drop_existing = [c for c in drop_cols if c in df.columns]
        df.drop(columns=drop_existing, inplace=True, errors="ignore")
        df.drop_duplicates(inplace=True)

    return df_46, df_79

df_46, df_79 = load_and_preprocess()
print(f"4-6: {df_46.shape}, 7-9: {df_79.shape}")
print(f"4-6 label: {df_46['label'].value_counts().to_dict()}")
print(f"7-9 label: {df_79['label'].value_counts().to_dict()}")

## 3.3 Preprocessing Helpers

Các hàm hỗ trợ xử lý missing values và chuẩn bị dữ liệu cho từng experiment.
- Numeric: điền median
- Categorical: điền `"__MISSING__"`


In [ ]:
def handle_missing(df):
    df = df.copy()
    for col in df.columns:
        if col == "label":
            continue
        if df[col].dtype == object:
            df[col] = df[col].where(pd.notna(df[col]), "__MISSING__")
        else:
            df[col] = df[col].fillna(df[col].median())
    return df

def encode_categoricals(df, encoders=None, fit=True):
    df = df.copy()
    if encoders is None:
        encoders = {}
    cat_cols = df.select_dtypes(include="object").columns.tolist()
    for col in cat_cols:
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        else:
            le = encoders.get(col)
            if le:
                mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
                df[col] = df[col].astype(str).map(mapping).fillna(-1).astype(int)
    return df, encoders

def scale_numeric(df, scaler=None, fit=True):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if "label" in numeric_cols:
        numeric_cols.remove("label")
    if fit:
        scaler = StandardScaler()
        df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    else:
        df[numeric_cols] = scaler.transform(df[numeric_cols])
    return df, scaler

def preprocess_df(df, encoders=None, scaler=None, fit=True):
    df = handle_missing(df)
    df, encoders = encode_categoricals(df, encoders, fit)
    df, scaler = scale_numeric(df, scaler, fit)
    return df, encoders, scaler

## 3.4 Resampling – SMOTE + RandomUnderSampler

Chiến lược kết hợp để xử lý mất cân bằng lớp (~1:40):

1. **SMOTE** (Synthetic Minority Over-sampling Technique): tạo mẫu synthetic cho lớp thiểu số, oversample đến **25%** của lớp đa số.
2. **RandomUnderSampler**: giảm lớp đa số xuống sao cho tỷ lệ majority:minority = **2:1** (sampling_strategy=0.5).

> Kết quả: tỷ lệ lớp cải thiện từ ~1:40 lên ~2:1, giúp model học tốt hơn pattern của lớp thiểu số mà không bị bias quá mức.


In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not installed. Skipping resampling.")

def get_resampled(X, y, smote_strategy=0.25, under_strategy=0.5, random_state=42):
    if not HAS_IMBLEARN:
        return X, y
    smote = SMOTE(sampling_strategy=smote_strategy, random_state=random_state)
    undersample = RandomUnderSampler(sampling_strategy=under_strategy, random_state=random_state)
    X_res, y_res = smote.fit_resample(X, y)
    X_res, y_res = undersample.fit_resample(X_res, y_res)
    print(f"Resampling: {pd.Series(y).value_counts().to_dict()} -> {pd.Series(y_res).value_counts().to_dict()}")
    return X_res, y_res

## 3.5 Experiment Building (α₁→α₄)

Xây dựng 8 cấu hình thí nghiệm từ 4 thiết kế chính:

| Experiment | Tên | Mô tả |
|------------|-----|-------|
| **α₁** | Train Future → Test Past | Train 7–9, test 4–6 (retroactive) |
| **α₂** | Train Past → Test Future | Train 4–6, test 7–9 (realistic deployment) |
| **α₃** | K-Fold Combined | Merge 4–6 + 7–9, 5-fold CV (có thể có leakage) |
| **α₄** | Incremental Learning | Train A + k% B, test (1−k)% B với k ∈ {10%,25%,50%,75%,90%} |

> α₂ được coi là thiết kế **realistic** nhất vì mô phỏng triển khai thực tế: train trên dữ liệu cũ, dự đoán dữ liệu mới.


In [ ]:
def build_alpha1(df_46, df_79):
    return {"name": "alpha1_Train79_Test46", "train": df_79.copy(), "test": df_46.copy()}

def build_alpha2(df_46, df_79):
    return {"name": "alpha2_Train46_Test79", "train": df_46.copy(), "test": df_79.copy()}

def build_alpha3(df_46, df_79):
    merged = pd.concat([df_46, df_79], ignore_index=True)
    merged = merged.sample(frac=1, random_state=config.RANDOM_STATE).reset_index(drop=True)
    return {"name": "alpha3_KFold_Combined", "train": merged, "test": None}

def build_alpha4(df_46, df_79, k_ratio):
    df_a = df_79.copy()  # period A = 7-9
    df_b = df_46.copy()  # period B = 4-6
    b_sample, b_test = train_test_split(
        df_b, test_size=(1 - k_ratio), stratify=df_b["label"], random_state=config.RANDOM_STATE
    )
    return {
        "name": f"alpha4_Incremental_{int(k_ratio*100)}pctB",
        "train": pd.concat([df_a, b_sample], ignore_index=True),
        "test": b_test.copy()
    }

experiments = [
    build_alpha1(df_46, df_79),
    build_alpha2(df_46, df_79),
    build_alpha3(df_46, df_79),
]
for r in config.INCREMENTAL_RATIOS:
    experiments.append(build_alpha4(df_46, df_79, r))

print(f"Built {len(experiments)} experiments:")
for e in experiments:
    print(f"  {e['name']}: train={e['train'].shape}, test={e['test'].shape if e['test'] is not None else 'internal'}")

### Lý do chọn các thiết kế thí nghiệm

- **α₁ (retroactive)**: Đánh giá xem model có thể "nhìn lại" và dự đoán quá khứ không. Thường cho kết quả tốt hơn vì dữ liệu future có nhiều thông tin hơn.
- **α₂ (realistic)**: Mô phỏng triển khai thực tế. Đây là metric quan trọng nhất để đánh giá khả năng production.
- **α₃ (combined)**: Tận dụng toàn bộ dữ liệu, nhưng có nguy cơ data leakage nếu các mẫu giữa 2 kỳ không độc lập.
- **α₄ (incremental)**: Đánh giá hiệu quả của việc bổ sung dần dữ liệu mới. Hữu ích khi triển khai model cần retrain định kỳ.


## 3.6 Model Training Functions

Factory function tạo classifier và hàm train với **Stratified K-Fold** (5 folds).

- **StratifiedKFold**: đảm bảo tỷ lệ `DELAY_FLG=1` giống nhau trong mỗi fold.
- **Early stopping**: LGBM dùng callback; XGB giữ `eval_set` (tương thích v3.2.0).
- **scale_pos_weight**: tự động tính cho XGB để cân bằng lớp.


In [ ]:
def get_classifier(model_name, y_train=None):
    if model_name == "LGBM":
        return lgb.LGBMClassifier(**config.LGBM_PARAMS)
    elif model_name == "XGB":
        params = config.XGB_PARAMS.copy()
        if y_train is not None:
            n_neg, n_pos = (y_train == 0).sum(), (y_train == 1).sum()
            params["scale_pos_weight"] = n_neg / n_pos if n_pos > 0 else 1.0
        return xgb.XGBClassifier(**params)
    elif model_name == "CatBoost":
        return CatBoostClassifier(**config.CAT_PARAMS)
    elif model_name == "KNN":
        return KNeighborsClassifier(**config.KNN_PARAMS)
    else:
        raise ValueError(f"Unknown model {model_name}")

def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
    }

def train_skfold(X_train, y_train, X_val, y_val, model_name, n_splits=config.N_SPLITS,
                 early_stopping=50, use_resampling=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=config.RANDOM_STATE)
    models = []
    fold_metrics = []
    oof_proba = np.zeros(len(X_train))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        if use_resampling:
            X_tr, y_tr = get_resampled(X_tr, y_tr)

        model = get_classifier(model_name, y_tr)

        if model_name == "LGBM":
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                      callbacks=[lgb.early_stopping(early_stopping, verbose=False)])
        elif model_name == "XGB":
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        elif model_name == "CatBoost":
            model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True, verbose=False)
        elif model_name == "KNN":
            model.fit(X_tr, y_tr)

        oof_proba[va_idx] = model.predict_proba(X_va)[:, 1]
        fold_met = compute_metrics(y_va, (oof_proba[va_idx] >= 0.5).astype(int), oof_proba[va_idx])
        fold_metrics.append(fold_met)
        models.append(model)
        print(f"  Fold {fold+1}/{n_splits} - ROC-AUC: {fold_met['roc_auc']:.4f}")

    final_model = models[0]
    val_proba = final_model.predict_proba(X_val)[:, 1]
    val_met = compute_metrics(y_val, (val_proba >= 0.5).astype(int), val_proba)

    return {
        "model_name": model_name,
        "final_model": final_model,
        "oof_metrics": compute_metrics(y_train, (oof_proba >= 0.5).astype(int), oof_proba),
        "val_metrics": val_met,
        "fold_metrics": pd.DataFrame(fold_metrics),
    }

def evaluate_on_test(model, X_test, y_test):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return compute_metrics(y_test, pred, proba)

## 3.7 GridSearchCV – Hyperparameter Tuning

Sử dụng `GridSearchCV` với **F1 scoring** và 3-fold CV để tìm hyperparameter tối ưu.
Hỗ trợ KNN, XGB, LGBM.

> F1 được chọn làm metric vì cân bằng giữa Precision và Recall — quan trọng với bài toán imbalance.


In [ ]:
def train_with_gridsearch(X_train, y_train, X_val, y_val, model_name,
                          param_grid=None, scoring="f1", cv=3):
    if model_name == "KNN":
        base = KNeighborsClassifier()
        if param_grid is None:
            param_grid = {"n_neighbors": [3, 5, 7, 9]}
    elif model_name == "XGB":
        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        base = xgb.XGBClassifier(eval_metric="auc", random_state=config.RANDOM_STATE,
                                 scale_pos_weight=neg / pos if pos > 0 else 1.0)
        if param_grid is None:
            param_grid = {
                "n_estimators": [100, 200],
                "max_depth": [5, 7, 10],
                "learning_rate": [0.05, 0.1],
            }
    elif model_name == "LGBM":
        base = lgb.LGBMClassifier(objective="binary", random_state=config.RANDOM_STATE)
        if param_grid is None:
            param_grid = {
                "n_estimators": [100, 200],
                "learning_rate": [0.05, 0.1],
                "max_depth": [5, 7, 10],
                "num_leaves": [15, 31],
            }
    else:
        raise ValueError(f"GridSearch not supported for {model_name}")

    grid = GridSearchCV(estimator=base, param_grid=param_grid, scoring=scoring,
                        cv=cv, verbose=1, n_jobs=-1)
    print(f"Start GridSearchCV for {model_name}...")
    grid.fit(X_train, y_train)

    print(f"Best parameters ({model_name}):", grid.best_params_)
    print(f"Best CV score ({scoring}):", round(grid.best_score_, 4))

    best_model = grid.best_estimator_
    val_pred = best_model.predict(X_val)
    val_proba = best_model.predict_proba(X_val)[:, 1]
    val_met = compute_metrics(y_val, val_pred, val_proba)

    return {
        "model_name": model_name,
        "best_params": grid.best_params_,
        "best_score": grid.best_score_,
        "final_model": best_model,
        "val_metrics": val_met,
    }

## 3.8 Run Experiments

Chạy toàn bộ 8 experiments × 4 models.
Vì dữ liệu lớn, ở đây demo với α₂ (Train 4–6, Test 7–9) làm ví dụ.

Kết quả được lưu vào `results/experiments/` dưới dạng CSV và confusion matrix plots.


In [ ]:
def run_experiment(exp, use_resampling=False, use_gridsearch=False):
    train_df = exp["train"].copy()
    test_df = exp["test"]

    # Preprocess train
    train_proc, encoders, scaler = preprocess_df(train_df, fit=True)
    y = train_proc["label"]
    X = train_proc.drop(columns=["label"])

    # Split 8:1:1
    X_trval, X_te, y_trval, y_te = train_test_split(
        X, y, test_size=config.TEST_RATIO, stratify=y, random_state=config.RANDOM_STATE
    )
    val_ratio_adj = config.VAL_RATIO / (1 - config.TEST_RATIO)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_trval, y_trval, test_size=val_ratio_adj,
        stratify=y_trval, random_state=config.RANDOM_STATE
    )

    # Determine test set
    if test_df is None:
        X_test, y_test = X_te, y_te
    else:
        test_proc, _, _ = preprocess_df(test_df, encoders=encoders, scaler=scaler, fit=False)
        X_test = test_proc.drop(columns=["label"])
        y_test = test_proc["label"]

    results = []
    models_to_run = ["LGBM", "XGB", "KNN"]  # CatBoost omitted for speed in demo

    for model_name in models_to_run:
        print(f"\n=== {exp['name']} | {model_name} ===")

        if use_gridsearch and model_name in ["KNN", "XGB", "LGBM"]:
            res = train_with_gridsearch(X_tr, y_tr, X_val, y_val, model_name)
        else:
            res = train_skfold(X_tr, y_tr, X_val, y_val, model_name, use_resampling=use_resampling)

        test_met = evaluate_on_test(res["final_model"], X_test, y_test)

        print("Test metrics:", {k: f"{v:.4f}" for k, v in test_met.items()})
        print("Confusion Matrix:")
        y_pred = (res["final_model"].predict_proba(X_test)[:, 1] >= 0.5).astype(int)
        print(confusion_matrix(y_test, y_pred))
        print("Classification Report:")
        print(classification_report(y_test, y_pred, zero_division=0, digits=4))

        row = {"experiment": exp["name"], "model": model_name}
        row.update({f"test_{k}": v for k, v in test_met.items()})
        row.update({f"val_{k}": v for k, v in res["val_metrics"].items()})
        results.append(row)

    return pd.DataFrame(results)

# Demo: alpha2 (Train 4-6, Test 7-9) with default settings
exp_demo = build_alpha2(df_46, df_79)
results_df = run_experiment(exp_demo, use_resampling=False, use_gridsearch=False)
print("\n=== Results Summary ===")
print(results_df.to_string(index=False))

## 3.9 Resampling Demo

So sánh hiệu quả khi dùng SMOTE + RandomUnderSampler.

Không resampling: model có xu hướng predict toàn bộ là class 0 → accuracy cao nhưng recall cho class 1 gần như bằng 0.
Có resampling: F1 và Recall cải thiện đáng kể, đánh đổi một chút Precision.


In [ ]:
print("\n=== Without Resampling ===")
res_no = run_experiment(exp_demo, use_resampling=False)

print("\n=== With SMOTE + Undersampling ===")
res_yes = run_experiment(exp_demo, use_resampling=True)

## 3.10 GridSearch Demo

Tìm hyperparameter tối ưu cho KNN và XGB.

Ví dụ với KNN:
- `n_neighbors`: [3, 5, 7]
- `weights`: ["uniform", "distance"]

Ví dụ với XGB:
- `max_depth`: [3, 5, 7]
- `learning_rate`: [0.01, 0.1]
- `n_estimators`: [100, 200]


In [ ]:
print("\n=== With GridSearchCV ===")
res_grid = run_experiment(exp_demo, use_gridsearch=True)

## 3.11 Cross-Period Evaluation

Đánh giá khả năng generalize qua các kỳ khác nhau:

- **α₁**: Train 7–9 → Test 4–6 (retroactive)
- **α₂**: Train 4–6 → Test 7–9 (realistic deployment)
- **α₃**: Combined K-Fold (tiềm ẩn leakage)
- **α₄**: Incremental learning với các tỷ lệ k% khác nhau

So sánh kết quả giữa các thiết kế giúp xác định model nào ổn định nhất và thiết kế nào phù hợp nhất cho production.


In [ ]:
all_results = []
for exp in experiments[:2]:  # Just alpha1 and alpha2 for demo
    print(f"\n{'='*60}")
    print(f"Running {exp['name']}")
    print(f"{'='*60}")
    res = run_experiment(exp, use_resampling=False)
    all_results.append(res)

full_results = pd.concat(all_results, ignore_index=True)
print("\n=== All Results ===")
print(full_results.to_string(index=False))

## 3.12 Save Results

Lưu kết quả experiments, classification reports, và confusion matrices.
Các file được tổ chức theo cấu trúc:
- `results/experiments/<exp_name>_<model>_report.csv`
- `results/experiments/<exp_name>_<model>_cm.png`


In [ ]:
os.makedirs(config.EXP_DIR, exist_ok=True)
full_results.to_csv(os.path.join(config.EXP_DIR, "model_results.csv"), index=False)
print("Saved results to", os.path.join(config.EXP_DIR, "model_results.csv"))